# 04. Resource Allocation & Policy Optimization

**Theme C: Data-Driven Resource Allocation of Insecticide-Treated Nets (ITNs)**  
**Target:** Ghana National Malaria Elimination Programme (NMEP)  
**Objective:** An equitable, uncertainty-aware ITN allocation across 50 northern districts under a strict constraint of 50,000 nets.

---

### Decision Framing Summary:
1. **Target:** Populations in northern Ghana vulnerable to malaria transmission.
2. **Unit of Analysis:** District level ($n = 50$ districts across Northern, Upper East and Upper West, 2014-17 boundaries).
3. **Objective Metric:** Prioritise unmet need: the upper 95% confidence bound of expected cases $\times$ the unmet coverage gap.
4. **Operational Constraints:** Exactly 50,000 integer nets, zero negative allocations, single shipment delivery.
5. **Equity vs. Efficiency Trade-off:** Reported cases also reflect testing and access, so a case-based rule favours districts that test more. Our rule treats everyone in a region as equally at risk (section 5 shows what that means).

In [ ]:
import sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure src/ can be imported regardless of execution working directory
root = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from scipy.stats import spearmanr
from src import io, models, uncertainty as unc, viz

RANDOM_SEED = io.RANDOM_SEED
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore')
viz.set_theme()

print('Setup complete. Random seed set to', RANDOM_SEED)

## 1. Load District Surveillance & Coverage Data
We load the curated district dataset covering the 50 northern districts.


In [ ]:
district = io.load_district_cases()
print(f"Loaded {len(district)} districts across {district['region_name'].nunique()} regions.")
district[['district', 'region_name', 'mean_population', 'positive_cases', 'net_coverage_pct']].head(5)


## 2. Fit Negative Binomial Model with Population Offset
As established in Theme A (`02_distributions.ipynb`, cells cd09 and cd25), district case counts are strongly over-dispersed (variance/mean 77,183), and a Poisson GLM is rejected even with a population offset (Pearson $\chi^2/\text{df} = 54{,}147$).

We fit a Negative Binomial regression with a $\log(\text{population})$ offset to estimate expected cases and their 95% confidence intervals.

In [ ]:
formula = "positive_cases ~ net_coverage_pct"
offset = np.log(district["mean_population"].clip(lower=1)).to_numpy()

# Fit working NB model
nb_fit = models.fit_negative_binomial_mle(formula, district, offset=offset)
print(f"Negative Binomial fit converged: {nb_fit.mle_retvals.get('converged')}")
print(f"Dispersion alpha (MLE): {float(nb_fit.params['alpha']):.4f}")


## 3. Allocation Policy: Case-Proportional Comparator vs. Equitable (Need-Weighted)

### The case-proportional comparator
Allocates nets in proportion to reported positive cases:
$$\text{Weight}_i^{\text{naive}} = \text{positive\_cases}_i$$
$$\text{Allocation}_i^{\text{naive}} = N \times \frac{\text{Weight}_i^{\text{naive}}}{\sum_j \text{Weight}_j^{\text{naive}}}$$

This is a comparator, not Ghana's current practice: national campaigns allocate ITNs by population (about one net per two people), plus antenatal and school channels (PMI Ghana Malaria Operational Plan FY2017).

*Weaknesses of the case-proportional rule:*
1. **Testing and access:** reported cases measure testing as well as malaria, so districts that test more receive more.
2. **Possible referral distortion:** regional hospitals may record patients from neighbouring districts. This is a hypothesis, checked in section 5.
3. **Ignores existing coverage:** it gives no weight to how many households already own a net.

---

### The equitable (need-weighted) policy
1. **Upper-bound expected cases:** $\hat{\mu}_{i,\text{upper}}$, the upper bound of the 95% confidence interval for district $i$'s expected cases from the Negative Binomial model. This is a confidence interval for the mean, not a prediction interval.
2. **Unmet coverage gap:** $g_i = 1 - \frac{\text{net\_coverage\_pct}_i}{100}$.
3. **Hamilton integer apportionment:** exactly 50,000 whole nets (`models.hamilton`).

$$\text{Weight}_i^{\text{equitable}} = \hat{\mu}_{i,\text{upper}} \times \left(1 - \frac{\text{coverage}_i}{100}\right)$$

In [ ]:
alloc_df = models.compute_allocation(district, nb_fit, total_nets=50000, offset=offset)

# Verify budget constraint
assert alloc_df['equitable_allocation'].sum() == 50000, "Equitable budget must sum to exactly 50,000"
assert alloc_df['naive_allocation'].sum() == 50000, "Naive budget must sum to exactly 50,000"

print(f"Budget verified: Exactly {alloc_df['equitable_allocation'].sum():,} nets allocated across {len(alloc_df)} districts.")
alloc_df.groupby('region_name')[['naive_allocation', 'equitable_allocation']].sum()

## 4. Empirical Evaluation: Top Gainers vs. Top Losers

Comparing the equitable allocation against the naive baseline reveals substantial policy reallocations ($\Delta = \text{Equitable} - \text{Naive}$).


In [ ]:
gainers = alloc_df.sort_values(by="delta", ascending=False).head(5)[
    ['district', 'region_name', 'mean_population', 'net_coverage_pct', 'naive_allocation', 'equitable_allocation', 'delta']
]
losers = alloc_df.sort_values(by="delta", ascending=True).head(5)[
    ['district', 'region_name', 'mean_population', 'net_coverage_pct', 'naive_allocation', 'equitable_allocation', 'delta']
]

print("=== TOP 5 GAINERS (Districts receiving MORE nets under equitable policy) ===")
print(gainers.to_string(index=False))
print()
print("=== TOP 5 LOSERS (Districts receiving FEWER nets under equitable policy) ===")
print(losers.to_string(index=False))


## 5. What the rule does in practice

`net_coverage_pct` takes one value per region, so expected cases are population times a regional rate.
Within a region, nets therefore follow population exactly, and the upper confidence bound adds one
multiplier per region. The coverage coefficient matters too: it scales expected risk, while the gap term
pulls the other way.

The table checks the districts the policy discussion names: reported cases per person (2014-17,
cumulative), rank within their region (1 = highest) and rank among all 50 (1 = lowest).

In [ ]:
practice = alloc_df.assign(
    nets_per_1000=1000 * alloc_df["equitable_allocation"] / alloc_df["mean_population"],
    upper_over_expected=alloc_df["pred_ci_upper"] / alloc_df["predicted_cases"],
    cases_per_person=alloc_df["positive_cases"] / alloc_df["mean_population"],
)
practice["rank_in_region"] = practice.groupby("region_code")["cases_per_person"].rank(ascending=False).astype(int)
practice["rank_of_50_lowest"] = practice["cases_per_person"].rank().astype(int)

coef, p = nb_fit.params["net_coverage_pct"], nb_fit.pvalues["net_coverage_pct"]
print(f"Coverage coefficient: {coef:+.4f} per point (p = {p:.1e}): higher-coverage regions had more cases")
for region, g in practice.groupby("region_name"):
    rho = spearmanr(g["delta"] / g["mean_population"], g["cases_per_person"])[0]
    print(f"{region}: {g.nets_per_1000.min():.2f} to {g.nets_per_1000.max():.2f} nets per 1,000; "
          f"upper/expected {g.upper_over_expected.iloc[0]:.3f}; "
          f"rank correlation of change per person with cases per person {rho:+.2f}")

named = ["Tamale", "Sagnarigu", "Bole", "Nabdam", "Bolgatanga", "Wa"]
cols = ["region_name", "cases_per_person", "rank_in_region", "rank_of_50_lowest", "delta"]
practice.set_index("district").loc[named, cols].round(2)

## 6. What would change the answer

One modelling choice changes at a time; the table compares regional totals with the proposal.
- **Point estimate** of expected cases instead of the upper confidence bound.
- **Region-effects model** (`positive_cases ~ C(region_code)`), which fits better by AIC.
- **Coverage corrected** for the 11 districts coded Northern that now lie in Savannah or North East
  (`io.REGION_2019`), using the survey-weighted coverage of those regions.
- **Northern coverage at its cluster-bootstrap bounds** (rounded to one decimal, as reported in
  notebook 02, cell cd39).

In [ ]:
mis = io.load_mis_sample()  # used for regional aggregates only


def owns_net(households):
    return unc.weighted_proportion(households, "has_net")


def northern_at(value):
    """District table with Northern Region's coverage replaced by value."""
    return district.assign(net_coverage_pct=district["net_coverage_pct"].where(district["region_code"] != 12, value))


assert set(io.REGION_2019) <= set(district["district"])
cov_2019 = {r: owns_net(mis[mis["region"] == r]) for r in (13, 14)}
corrected = district.assign(net_coverage_pct=district["district"].map(io.REGION_2019).map(cov_2019)
                            .fillna(district["net_coverage_pct"]))
_, lo, hi = unc.cluster_bootstrap_ci(mis[mis["region"] == 12], owns_net, n_bootstraps=2000, random_seed=RANDOM_SEED)
print(f"Survey coverage: Savannah {cov_2019[13]:.1f}%, North East {cov_2019[14]:.1f}%; "
      f"Northern cluster-bootstrap bounds {lo:.1f}% to {hi:.1f}%")
nb_region = models.fit_negative_binomial_mle("positive_cases ~ C(region_code)", district, offset=offset)

scenarios = {
    "As proposed": alloc_df["equitable_allocation"],
    "Point estimate instead of upper bound": models.hamilton(alloc_df["predicted_cases"] * alloc_df["unmet_need_gap"], 50000),
    f"Region-effects model (AIC {nb_region.aic:,.1f} vs {nb_fit.aic:,.1f})":
        models.compute_allocation(district, nb_region, offset=offset)["equitable_allocation"],
    "Coverage corrected for 11 districts": models.allocate(corrected)["equitable_allocation"],
    f"Northern coverage at {lo:.1f}%": models.allocate(northern_at(round(lo, 1)))["equitable_allocation"],
    f"Northern coverage at {hi:.1f}%": models.allocate(northern_at(round(hi, 1)))["equitable_allocation"],
}
base = alloc_df["equitable_allocation"].to_numpy()
table = pd.DataFrame({name: pd.Series(np.asarray(v), index=district.index).groupby(district["region_name"]).sum()
                      for name, v in scenarios.items()}).T
table["nets_moved"] = [int(np.abs(np.asarray(v) - base).sum() // 2) for v in scenarios.values()]
table

## 7. Visualizing Policy Shifts
We visualize the reallocations using a comparative horizontal delta bar chart, exported for the Ministry presentation deck.

In [ ]:
# 1. Bar Chart: Top Gainers and Losers
fig1, ax1 = viz.plot_allocation_comparison(alloc_df, n_top=5)
fig1_path = viz.save_figure(fig1, 'c1_allocation_comparison.png')
print(f'Saved bar chart: {fig1_path}')
plt.show()

# 2. Map Highlight 1: Equitable Allocation Choropleth
root = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
adm0_p = root / 'data' / 'ghana_boundaries' / 'gha_admin0.geojson'
adm1_p = root / 'data' / 'ghana_boundaries' / 'gha_admin1.geojson'
adm2_p = root / 'data' / 'ghana_boundaries' / 'gha_admin2.geojson'

fig2, ax2 = viz.plot_district_choropleth(
    adm0_p, adm1_p, adm2_p, alloc_df,
    value_col='equitable_allocation',
    title='Equitable ITN Allocation (50,000 Nets) across Northern Ghana',
    legend_label='Nets allocated under the equitable rule'
)
fig2_path = viz.save_figure(fig2, 'c2_allocation_map.png')
print(f'Saved allocation map: {fig2_path}')
plt.show()

# 3. Map Highlight 2: Policy Shift (Gainers vs Losers)
fig3, ax3 = viz.plot_policy_shift_map(adm1_p, adm2_p, alloc_df)
fig3_path = viz.save_figure(fig3, 'c3_policy_shift_map.png')
print(f'Saved policy shift map: {fig3_path}')
plt.show()

## 8. Policy Recommendations & Oral Defense Takeaways

1. **Within each region, the rule allocates by population.** Coverage is one number per region, so every district in a region gets the same nets per person (section 5). That is close to Ghana's own population-based campaigns; say so openly.
2. **Why Tamale (+1,692) and Sagnarigu (+1,012) gain:** they are the two most populous districts, and Tamale reports the fewest cases per person of all 50 (section 5). A case-based rule would give them little; ours treats their residents as being at the same risk as the rest of Northern Region. That is a value judgement, not a correction.
3. **Why Wa (-1,516) and Bolgatanga (-665) lose:** the rule moves nets towards Northern Region overall, and both districts rank high in their regions on cases per person (Wa 1st of 11, Bolgatanga 4th of 13), which a population rule does not reward. Referral bias may add to this, but it is a hypothesis: Nabdam, often named as a sending district, ranks 1st in Upper East.
4. **The weakest part is the split between regions:** coverage is from 2022 and regional, its coefficient is positive (nets went where malaria was worst), and a better-fitting region-effects model moves 7,720 nets (section 6).